In [1]:
import json
import pandas as pd
from typing import Any, Dict
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# Creación de un driver general para todo el código
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")

# Servicio con chrome driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

In [3]:
# Definimos una función para entrar al link usando un driver y devolver un JSON (dict)
def page_scraper(driver, page_url: str, timeout: int = 10) -> Dict[str, Any]:

    # Usando el driver, entramos a la página - definimos un timeout en segundos por si se tarda
    driver.get(page_url)
    pre = WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.TAG_NAME, "pre")))

    # Convertimos los datos en formato JSON
    data_json = json.loads(pre.text)
    return(data_json)

In [4]:
# Dada una liga y temporada, obtenemos información general de los partidos (equipos y partidos)
def get_matches_information(driver, league_code: int, season_code: int) -> pd.DataFrame:

    # Lista a la cual vamos a ir concatenando la información
    matches_info = []

    # Para inicializar
    i = 0

    # Obtenemos la información -> esta esta dividida en páginas, por lo tanto, mientras se pueda buscar, obtenemos información
    while True:

        # Definimos url y scrapeamos usando la función
        season_part_url = f'https://api.sofascore.com/api/v1/unique-tournament/{league_code}/season/{season_code}/events/last/{i}'
        scraped_info = page_scraper(driver, season_part_url)

        # Entramos a los eventos, si no existen, terminamos
        matches_event_data = scraped_info.get('events')
        if not matches_event_data:
            break

        # Cada información scrapeada tiene distintos partidos
        for event_data in matches_event_data:

            # Elegimos la información que nos interesa
            event_data_info = {'id': event_data.get('id'),
                               'tournament': event_data.get('tournament').get('name'),
                               'season': event_data.get('season').get('year'),
                               'round': event_data.get('roundInfo').get('round'),
                               'status': event_data.get('status').get('description'),
                               'home_team': event_data.get('homeTeam').get('name'),
                               'away_team': event_data.get('awayTeam').get('name'),
                               'home_score': event_data.get('homeScore').get('current'),
                               'away_score': event_data.get('awayScore').get('current')}
            
            # Concatenamos a la lista
            matches_info.append(event_data_info)

        i += 1
    
    # Returnamos como dataframe la información de los partidos
    return pd.DataFrame(matches_info).sort_values(by='round')               # Ordenamos por ronda

In [5]:
# Dado el identificador de un partido, obtenemos información principal relevante (no entramos en estadísticas) - obtenemos más información que anteriormente
def get_match_more_information(driver, match_id: int) -> dict:

    # Information
    match_link = f'https://api.sofascore.com/api/v1/event/{match_id}'
    match_data = page_scraper(driver, match_link)
    match_data = match_data.get('event')

    # Ordenamos la información de nuestro partido en formato de diccionario y lo devolvemos
    return {'id': match_data.get('id'),
            'venue': match_data.get('venue', {}).get('stadium', {}).get('name') if match_data.get('venue') else None,
            'attendance': match_data.get('attendance'),
            'referee': match_data.get('referee', {}).get('name') if match_data.get('referee') else None,
            'home_manager': match_data.get('homeTeam', {}).get('manager', {}).get('name') if match_data.get('homeTeam', {}).get('manager') else None,
            'away_manager': match_data.get('awayTeam', {}).get('manager', {}).get('name') if match_data.get('awayTeam', {}).get('manager') else None}

In [6]:
# A partir del ID del partido, obtiene las estadísticas de los equipos
def get_teams_statistics(driver, match_id: int, home_team: str, away_team: str) -> pd.DataFrame:

    # A partir del link, obtenemos las estadísticas
    match_stats_link = f'https://api.sofascore.com/api/v1/event/{match_id}/statistics'
    match_stats = page_scraper(driver, match_stats_link)

    # Verificamos si existen estadísticas
    if 'statistics' not in match_stats or not match_stats['statistics']:
        # Si no hay estadísticas, devolvemos diccionarios vacíos
        dict_statistics_home = {'match': match_id, 'team': home_team, 'home_away': 'Home'}
        dict_statistics_away = {'match': match_id, 'team': away_team, 'home_away': 'Away'}
        return dict_statistics_home, dict_statistics_away

    # Obtenemos las estadísticas generales de TODO (0) el partido
    match_stats = match_stats['statistics'][0]['groups']

    # Diccionario para cada equipo
    dict_statistics_home = {'match': match_id, 'team': home_team, 'home_away': 'Home'}
    dict_statistics_away = {'match': match_id, 'team': away_team, 'home_away': 'Away'}

    # Concatenamos las estadísticas
    for statistic_type in match_stats:

        # Obtenemos los items
        statistics_items = statistic_type['statisticsItems']

        # Para cada item, añadimos la información
        for item in statistics_items:
            dict_statistics_home[item.get('key')] = item.get('homeValue')
            dict_statistics_away[item.get('key')] = item.get('awayValue')

    # Concatenamos los dos diccionarios y los devolvemos
    return dict_statistics_home, dict_statistics_away

In [7]:
# A partir del ID del partido, obtenemos un dataframe con las estadísticas de los jugadores 
def get_players_statistics(driver, match_id: int, home_team: str, away_team: str) -> pd.DataFrame:
    
    # Obtenemos el URL y lo scrapeamos
    match_lineups_link = f'https://api.sofascore.com/api/v1/event/{match_id}/lineups'
    match_lineups = page_scraper(driver, match_lineups_link)

    # Verificamos si existen alineaciones
    if 'home' not in match_lineups or 'away' not in match_lineups:
        return [], None, None

    # Diferenciamos entre alineaciones locales y visitantes
    home_lineups = match_lineups.get('home')
    away_lineups = match_lineups.get('away')

    # Obtenemos la formación de los dos equipos
    home_formation = home_lineups.get('formation')
    away_formation = away_lineups.get('formation')

    # Obtenemos los jugadores
    home_players = home_lineups.get('players', [])
    away_players = away_lineups.get('players', [])

    # Creamos una lista para añadir la información y las estadísticas de los jugadores (las pondremos en diccionarios individuales)
    home_players_info_stats = []
    away_players_info_stats = []

    # Para cada jugador local
    for player in home_players:

        # Creamos un diccionario para añadir, primero, la información
        player_dict = {'match': match_id,
                    'team': home_team,
                    'player': player.get('player').get('name'),
                    'shirt_number': player.get('shirtNumber'),
                    'position': player.get('position'),
                    'starter': False if player.get('substitute') else True}
        
        # Obtenemos las estadísticas
        player_stats = player.get('statistics')

        # Definimos todas las keys que vamos a mirar, quitamos algunos valores
        player_metrics = [k for k in player_stats.keys() if k not in ('ratingVersions', 'statisticsType')]

        # Para cada metrica, añadimos el valor
        for metric in player_metrics:
            player_dict[metric] = player_stats[metric]

        # Concatenamos
        home_players_info_stats.append(player_dict)

    # Lo mismo para el equipo visitante
    for player in away_players:

        player_dict = {'match': match_id,
                    'team': away_team,
                    'player': player.get('player').get('name'),
                    'shirt_number': player.get('shirtNumber'),
                    'position': player.get('position'),
                    'starter': False if player.get('substitute') else True}
        
        player_stats = player.get('statistics')
        player_metrics = [k for k in player_stats.keys() if k not in ('ratingVersions', 'statisticsType')]

        for metric in player_metrics:
            player_dict[metric] = player_stats[metric]

        away_players_info_stats.append(player_dict)

    # Unimos las dos listas y returnamos
    return (home_players_info_stats + away_players_info_stats), home_formation, away_formation

In [8]:
# A partir del ID del partido y los equpos, obtenemos un dataframe con los tiros
def get_match_shotmap(driver, match_id: int, home_team: str, away_team: str) -> pd.DataFrame:

    # Obtenemos URL y scrapeamos
    match_shotmap_link = f'https://api.sofascore.com/api/v1/event/{match_id}/shotmap'
    match_shotmap = page_scraper(driver, match_shotmap_link)

    # Verificamos si existe shotmap
    if 'shotmap' not in match_shotmap:
        return []

    # Lista para ir añadiendo información
    shots_list = []

    # Para cada tiro, creamos un diccionario con su información
    for shot in match_shotmap['shotmap']:

        # Obtenemos coordenadas
        player_coords = shot.get("playerCoordinates") or {}
        goal_coords = shot.get("goalMouthCoordinates") or {}
        block_coords = shot.get("blockCoordinates") or {}

        # Información a un diccionario
        shot_dict = {'match': match_id,
                    'team': home_team if shot.get('isHome') else away_team,
                    'player': shot.get('player').get('name'),
                    'type': shot.get('shotType'),
                    'situation': shot.get('situation'),
                    'time_seconds': shot.get('timeSeconds'), 
                    'start_coord_x': shot.get('playerCoordinates').get('x'),
                    "start_coord_x": player_coords.get("x"),
                    "start_coord_y": player_coords.get("y"),
                    "start_coord_z": player_coords.get("z"),
                    "goal_coord_x": goal_coords.get("x"),
                    "goal_coord_y": goal_coords.get("y"),
                    "goal_coord_z": goal_coords.get("z"),
                    "block_coord_x": block_coords.get("x"),
                    "block_coord_y": block_coords.get("y"),
                    "block_coord_z": block_coords.get("z")}
        
        # Añadimos a la lista
        shots_list.append(shot_dict)

    return shots_list

In [9]:
# Función principal de scrapeo de una liga
def full_league_scraping(league_code: int, season_code: int, slug: str) -> None:

    # Leemos los CSVs con información de partidos anteriores
    data_path = Path("data/raw/")
    hist_match_info = pd.read_csv(data_path / "MatchesInformation.csv")
    hist_team_stats = pd.read_csv(data_path / "TeamStatistics.csv")
    hist_player_stats = pd.read_csv(data_path / "PlayerStatistics.csv")
    hist_shot_map = pd.read_csv(data_path / "ShotMap.csv")

    # Creamos el directorio de salida si no existe
    output_dir = Path(f"data/{slug}")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Comprovamos aquellos partidos que hayan sido scrapeados -> aquellos que salgan en todas las listas
    matches_1 = hist_match_info['id'].unique().tolist()
    matches_2 = hist_team_stats['match'].unique().tolist()
    matches_3 = hist_player_stats['match'].unique().tolist()
    matches_4 = hist_shot_map['match'].unique().tolist()

    # Creamos una lista con los partidos scrapeados
    scraped_matches = list(set(matches_1) & set(matches_2) & set(matches_3) & set(matches_4))

    # Obtenemos la información de los partidos
    matches_information = get_matches_information(driver, league_code=league_code, season_code=season_code)

    # Listas para añadir toda la información que tenemos
    more_information_list = []
    teams_statistics_list = []
    players_statistics_list = []
    shot_map_list = []

    # Para imprimir
    i = 1
    total_matches = len(matches_information) - len(scraped_matches)

    # Para cada partido, buscamos información
    for index, match in matches_information.iterrows():

        # Información del partido
        match_id = match['id']
        home_team = match['home_team']
        away_team = match['away_team']

        # Comprovamos que el partido no se haya scrapeado antes; en caso de que se haya scrapeado, saltamos
        if match_id in scraped_matches:
            continue

        # Imprimimos para informar
        print(f'Scrapeando información del partido {match_id} ({home_team} vs {away_team}) [{i}/{total_matches}]')
        i += 1

        # Obtenemos la información que deseamos usando las funciones creadas
        more_information = get_match_more_information(driver, match_id=match_id)
        home_team_statistics, away_team_statistics = get_teams_statistics(driver, match_id=match_id, home_team=home_team, away_team=away_team)
        players_statistics, home_formation, away_formation = get_players_statistics(driver, match_id=match_id, home_team=home_team, away_team=away_team)
        match_shot_map = get_match_shotmap(driver, match_id=match_id, home_team=home_team, away_team=away_team)

        # Añadimos a more information la formación de cada equipo
        more_information['home_formation'] = home_formation
        more_information['away_formation'] = away_formation

        # Añadimos a las listas
        more_information_list.append(more_information)
        teams_statistics_list.append(home_team_statistics)
        teams_statistics_list.append(away_team_statistics)
        players_statistics_list = players_statistics_list + players_statistics      # En este caso concatenamos
        shot_map_list = shot_map_list + match_shot_map

        # Transformamos en dataframe
        more_information_df = pd.DataFrame(more_information_list)
        teams_statistics_df = pd.DataFrame(teams_statistics_list)
        players_statistics_df = pd.DataFrame(players_statistics_list)
        shot_map_df = pd.DataFrame(shot_map_list)

        # Concatenamos los dataframes de información
        more_information_df = matches_information.merge(more_information_df, on='id')

        # Concatenamos los dataframes historicos con los actuales
        more_information_df = pd.concat([hist_match_info, more_information_df])
        teams_statistics_df = pd.concat([hist_team_stats, teams_statistics_df])
        players_statistics_df = pd.concat([hist_player_stats, players_statistics_df])
        shot_map_df = pd.concat([hist_shot_map, shot_map_df])

        # Guardamos en formato CSV -> guardado dentro del bucle para no perder información
        matches_information.to_csv(output_dir / "MatchesInformation.csv", index=False)
        teams_statistics_df.to_csv(output_dir / "TeamStatistics.csv", index=False)
        players_statistics_df.to_csv(output_dir / "PlayerStatistics.csv", index=False)
        shot_map_df.to_csv(output_dir / "ShotMap.csv", index=False)

        # Esperamos 3 segundos para seguir con el siguiente
        time.sleep(3)

In [10]:
# Codigo de liga y temporada que queremos obtener
serie_a_code = 23
season_2526_code = 76457
slug = 'serie_a_2526'

# Aplicación de la función
full_league_scraping(league_code=serie_a_code, season_code=season_2526_code, slug=slug)

Scrapeando información del partido 13981568 (Fiorentina vs Udinese) [1/155]
Scrapeando información del partido 13981572 (Lazio vs Cremonese) [2/155]
Scrapeando información del partido 13981580 (Sassuolo vs Torino) [3/155]
Scrapeando información del partido 13981581 (Cagliari vs Pisa) [4/155]
Scrapeando información del partido 13981576 (Hellas Verona vs Bologna) [5/155]
Scrapeando información del partido 13981578 (Napoli vs Parma) [6/155]
Scrapeando información del partido 13981574 (Inter vs Lecce) [7/155]
Scrapeando información del partido 14844287 (Inter vs Lecce) [8/155]
Scrapeando información del partido 14844295 (Napoli vs Parma) [9/155]
Scrapeando información del partido 14844293 (Hellas Verona vs Bologna) [10/155]
Scrapeando información del partido 13981579 (Genoa vs Atalanta) [11/155]
Scrapeando información del partido 13981567 (Como vs Milan) [12/155]
Scrapeando información del partido 14844296 (Como vs Milan) [13/155]
Scrapeando información del partido 13981575 (Juventus vs Ro